# 17 - V2 continual prototype-herding evaluation

Compare fine-tuning, reservoir replay, and balanced prototype-herding memory with validation-gated imprinting.


In [1]:
from pathlib import Path
import os, sys

LOCAL_CANDIDATES = [
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]

def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').is_file() and (path / 'src' / 'rareguard').is_dir()

def find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, *LOCAL_CANDIDATES, *DRIVE_CANDIDATES]:
        if is_project_root(candidate):
            return candidate
    if Path('/content').is_dir():
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        for candidate in DRIVE_CANDIDATES:
            if is_project_root(candidate):
                return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError('Canonical RareGuard-XIDS IMPLEMENTATION project not found.')
os.chdir(PROJECT_ROOT)
PROJECT_SRC = str(PROJECT_ROOT / 'src')
sys.path.insert(0, PROJECT_SRC)
os.environ['PYTHONPATH'] = PROJECT_SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Project root:', PROJECT_ROOT)
print('Runtime:', 'Colab' if Path('/content').is_dir() else 'local')
print('Python path ready')


Project root: /content/gdrive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Runtime: Colab
Python path ready


In [2]:
import subprocess
import torch

DEVELOPMENT_DATASETS = [
    'cse_cic_ids2018',
    'edge_iiotset',
    'cic_bccc_nrc_tabulariotattack_2024',
    'cic_iiot_2025_datasense',
]
CONFIRMATION_DATASETS = ['ciciot2023', 'ton_iot']

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA runtime is required for the development-dataset stage.')

def run_device_split_stage(module: str) -> None:
    for dataset in DEVELOPMENT_DATASETS:
        subprocess.run(
            [sys.executable, '-m', module, '--config', 'configs/publication_v2.yaml',
             '--datasets', 'configs/datasets_v2.yaml', '--dataset', dataset,
             '--device', 'cuda', '--resume'],
            check=True,
        )
    for dataset in CONFIRMATION_DATASETS:
        subprocess.run(
            [sys.executable, '-m', module, '--config', 'configs/publication_v2.yaml',
             '--datasets', 'configs/datasets_v2.yaml', '--dataset', dataset,
             '--device', 'cpu', '--resume'],
            check=True,
        )

run_device_split_stage('rareguard.experiments.run_publication_continual')


In [3]:
import pandas as pd
display(pd.read_csv('artifacts/publication_v2/results/continual_seed_metrics.csv'))


,dataset,seed,method,tasks,memory_size,mean_validation_forgetting,max_validation_forgetting,training_seconds,test_seconds,test_samples_per_second,...,f1_macro,f1_weighted,balanced_accuracy,mcc,cohen_kappa,ece_15,rare_class_count,rare_f1_macro,roc_auc_macro_ovr,pr_auc_macro
0,cse_cic_ids2018,42,RareGuard-FineTune,4,0,0.093863,0.375454,4.151590,0.058688,72041.431116,...,0.492784,0.601222,0.581246,0.620499,0.591290,0.167821,3.0,0.208914,0.977510,0.745008
1,cse_cic_ids2018,42,RareGuard-ReservoirReplay,4,1500,0.284943,1.000000,2.821997,0.062740,67388.855885,...,0.740007,0.927491,0.817483,0.916192,0.915888,0.017252,3.0,0.250712,0.979500,0.834908
2,cse_cic_ids2018,42,RareGuard-Continual,4,1500,0.056400,0.225600,3.813029,0.058595,72156.409839,...,0.643680,0.876102,0.763229,0.842484,0.840832,0.062139,3.0,0.127049,0.982921,0.756543
3,cse_cic_ids2018,73,RareGuard-FineTune,4,0,0.175278,0.571429,2.301119,0.057569,73441.812866,...,0.709184,0.898383,0.760737,0.880648,0.878530,0.044868,3.0,0.258905,0.989191,0.761403
4,cse_cic_ids2018,73,RareGuard-ReservoirReplay,4,1500,0.428698,0.800000,2.807167,0.057146,73985.390894,...,0.723887,0.905201,0.778503,0.892200,0.890910,0.036413,3.0,0.271132,0.982260,0.797763
5,cse_cic_ids2018,73,RareGuard-Continual,4,1500,0.183829,0.571429,3.797410,0.056937,74257.613113,...,0.740407,0.904824,0.804353,0.873269,0.871480,0.038952,3.0,0.261907,0.988281,0.825170
6,cse_cic_ids2018,101,RareGuard-FineTune,4,0,0.076207,0.162393,2.300471,0.065660,64391.896514,...,0.698851,0.869626,0.800613,0.861587,0.857964,0.066264,3.0,0.298137,0.992635,0.843771
7,cse_cic_ids2018,101,RareGuard-ReservoirReplay,4,1500,0.119608,0.333333,2.795274,0.063260,66835.771074,...,0.764606,0.927806,0.817484,0.917442,0.917136,0.026605,3.0,0.343434,0.994432,0.845078
8,cse_cic_ids2018,101,RareGuard-Continual,4,1500,0.309419,1.000000,3.731078,0.058145,72715.366344,...,0.675510,0.899096,0.774225,0.871461,0.869969,0.047162,3.0,0.130556,0.990473,0.840607
9,edge_iiotset,42,RareGuard-FineTune,4,0,0.169440,0.510888,4.258350,0.048305,72103.923553,...,0.410541,0.534135,0.538018,0.525207,0.489311,0.104671,1.0,0.011561,0.953772,0.712855
